In [7]:
from abc import ABC, abstractmethod
from typing import Any, List, Tuple

class Problem(ABC):
    """Abstract base class for a search problem."""

    @abstractmethod
    def initial_state(self) -> Any:
        pass

    @abstractmethod
    def is_goal(self, state: Any) -> bool:
        pass

    @abstractmethod
    def actions(self, state: Any) -> List[Any]:
        pass

    @abstractmethod
    def result(self, state: Any, action: Any) -> Any:
        pass

    @abstractmethod
    def action_cost(self, state: Any, action: Any, next_state: Any) -> float:
        pass

In [8]:
#lab 1
MOVES = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1),
}


class GridProblem(Problem):
    def __init__(
        self,
        grid: List[List[int]],
        start: Tuple[int, int],
        goal: Tuple[int, int],
    ):
        """
        grid:
            2D list where 0 = free cell and 1 = obstacle.

        start, goal:
            Tuples in the form (row, col).
        """
        self.grid = grid
        self.start = start
        self.goal = goal

        self.rows = len(grid)
        self.cols = len(grid[0])

    def initial_state(self) -> Tuple[int, int]:
        return self.start

    def is_goal(self, state: Tuple[int, int]) -> bool:
        # TODO 1:
        return state == self.goal

    def in_bounds(self, state: Tuple[int, int]) -> bool:
        row, col = state
        return 0 <= row < self.rows and 0 <= col < self.cols

    def is_free(self, state: Tuple[int, int]) -> bool:
        row, col = state
        return self.grid[row][col] == 0

    def actions(self, state: Tuple[int, int]) -> List[str]:
        # TODO 2:
        
        legal_actions = []

        row, col = state

        for action, (dr, dc) in MOVES.items():
            next_state = (row + dr, col + dc)

            if self.in_bounds(next_state) and self.is_free(next_state):
                legal_actions.append(action)

        return legal_actions

    def result(self, state: Tuple[int, int], action: str) -> Tuple[int, int]:
        # TODO 3:
        
        row, col = state
        dr, dc = MOVES[action]
        return (row + dr, col + dc)

    def action_cost(
        self,
        state: Tuple[int, int],
        action: str,
        next_state: Tuple[int, int],
    ) -> float:
       
        return 1


In [12]:
from dataclasses import dataclass
from typing import Any, Optional, List, Dict

@dataclass
class Node:
    state: Any
    parent: Optional["Node"] = None
    action: Optional[Any] = None
    path_cost: float = 0
    depth: int = 0

    def __post_init__(self):
        if self.parent is not None:
            self.depth = self.parent.depth + 1


@dataclass
class SearchResult:
    algorithm: str
    status: str
    solution: Optional[Node]
    nodes_expanded: int
    max_frontier_size: int
    reached_count: int = 0
    limit: Optional[int] = None
    iterations: Optional[List[Dict[str, Any]]] = None

    @property
    def path(self) -> Optional[List[Any]]:
        if self.solution is None:
            return None
        return reconstruct_path(self.solution)

    @property
    def solution_depth(self) -> Optional[int]:
        if self.solution is None:
            return None
        return self.solution.depth

    @property
    def solution_cost(self) -> Optional[float]:
        if self.solution is None:
            return None
        return self.solution.path_cost

In [14]:
from typing import Iterable
class SearchAlgorithm(ABC):
    """Base class for search algorithms."""

    def expand(self, problem: Problem, node: Node) -> Iterable[Node]:
        # TODO 5:
        state = node.state

        for action in problem.actions(state):
            next_state = problem.result(state, action)
            cost = node.path_cost + problem.action_cost(state, action, next_state)

            yield Node(
                state=next_state,
                parent=node,
                action=action,
                path_cost=cost,
            )
    @abstractmethod
    def search(self, problem: Problem) -> SearchResult:
        pass

In [30]:
from collections import deque

class BreadthFirstSearch(SearchAlgorithm):
    def search(self, problem: Problem) -> SearchResult:
        algorithm = "BFS"

        start_node = Node(problem.initial_state())

        if problem.is_goal(start_node.state):
            return SearchResult(
                algorithm=algorithm,
                status="success",
                solution=start_node,
                nodes_expanded=0,
                max_frontier_size=1,
                reached_count=1,
            )

        frontier = deque([start_node])
        reached = {start_node.state}
        nodes_expanded = 0
        max_frontier_size = 1

        while frontier:
            node = frontier.popleft()
            nodes_expanded += 1

            for child in self.expand(problem, node):
                if child.state not in reached:
                    reached.add(child.state)

                    if problem.is_goal(child.state):
                        return SearchResult(
                            algorithm=algorithm,
                            status="success",
                            solution=child,
                            nodes_expanded=nodes_expanded,
                            max_frontier_size=max_frontier_size,
                            reached_count=len(reached),
                        )

                    frontier.append(child)
                    max_frontier_size = max(max_frontier_size, len(frontier))

        return SearchResult(
            algorithm=algorithm,
            status="failure",
            solution=None,
            nodes_expanded=nodes_expanded,
            max_frontier_size=max_frontier_size,
            reached_count=len(reached),
        )

In [32]:
class DepthFirstSearch(SearchAlgorithm):
    def search(self, problem: Problem) -> SearchResult:
        algorithm = "DFS"

        start_node = Node(problem.initial_state())

        if problem.is_goal(start_node.state):
            return SearchResult(
                algorithm=algorithm,
                status="success",
                solution=start_node,
                nodes_expanded=0,
                max_frontier_size=1,
                reached_count=1,
            )

        frontier = [start_node]
        reached = {start_node.state}
        nodes_expanded = 0
        max_frontier_size = 1

        while frontier:
            node = frontier.pop()
            nodes_expanded += 1

            for child in self.expand(problem, node):
                if child.state not in reached:
                    reached.add(child.state)

                    if problem.is_goal(child.state):
                        return SearchResult(
                            algorithm=algorithm,
                            status="success",
                            solution=child,
                            nodes_expanded=nodes_expanded,
                            max_frontier_size=max_frontier_size,
                            reached_count=len(reached),
                        )

                    frontier.append(child)
                    max_frontier_size = max(max_frontier_size, len(frontier))

        return SearchResult(
            algorithm=algorithm,
            status="failure",
            solution=None,
            nodes_expanded=nodes_expanded,
            max_frontier_size=max_frontier_size,
            reached_count=len(reached),
        )

In [20]:
class DepthLimitedSearch(SearchAlgorithm):
    def search(self, problem: Problem, limit: int = 10) -> SearchResult:
        algorithm = "DLS"

        initial_node = Node(problem.initial_state())

        metrics = {
            "nodes_expanded": 0,
            "max_stack_size": 1,
        }

        solution, status = self._recursive_dls(
            problem=problem,
            node=initial_node,
            limit=limit,
            metrics=metrics,
            current_stack_size=1,
        )

        return SearchResult(
            algorithm=algorithm,
            status=status,
            solution=solution,
            nodes_expanded=metrics["nodes_expanded"],
            max_frontier_size=metrics["max_stack_size"],
            reached_count=0,
            limit=limit,
        )

    def _recursive_dls(
        self,
        problem: Problem,
        node: Node,
        limit: int,
        metrics: Dict[str, int],
        current_stack_size: int,
    ) -> Tuple[Optional[Node], str]:

        if problem.is_goal(node.state):
            return node, "success"

        if limit == 0:
            return None, "cutoff"

        metrics["nodes_expanded"] += 1

        cutoff_occurred = False

        for child in self.expand(problem, node):
            metrics["max_stack_size"] = max(
                metrics["max_stack_size"],
                current_stack_size + 1
            )

            result, status = self._recursive_dls(
                problem=problem,
                node=child,
                limit=limit - 1,
                metrics=metrics,
                current_stack_size=current_stack_size + 1,
            )

            if status == "cutoff":
                cutoff_occurred = True
            elif status == "success":
                return result, "success"

        if cutoff_occurred:
            return None, "cutoff"

        return None, "failure"

In [22]:
class IterativeDeepeningSearch(SearchAlgorithm):
    def search(self, problem: Problem, max_depth: int = 50) -> SearchResult:
        algorithm = "IDS"

        total_expanded = 0
        max_frontier_size = 0
        iteration_log = []

        for depth in range(max_depth + 1):
            dls = DepthLimitedSearch()
            result = dls.search(problem, limit=depth)

            total_expanded += result.nodes_expanded
            max_frontier_size = max(max_frontier_size, result.max_frontier_size)

            iteration_log.append({
                "limit": depth,
                "status": result.status,
                "nodes_expanded": result.nodes_expanded,
            })

            if result.status == "success":
                return SearchResult(
                    algorithm=algorithm,
                    status="success",
                    solution=result.solution,
                    nodes_expanded=total_expanded,
                    max_frontier_size=max_frontier_size,
                    reached_count=0,
                    limit=depth,
                    iterations=iteration_log,
                )

            if result.status == "failure":
                return SearchResult(
                    algorithm=algorithm,
                    status="failure",
                    solution=None,
                    nodes_expanded=total_expanded,
                    max_frontier_size=max_frontier_size,
                    reached_count=0,
                    limit=depth,
                    iterations=iteration_log,
                )

        return SearchResult(
            algorithm=algorithm,
            status="cutoff",
            solution=None,
            nodes_expanded=total_expanded,
            max_frontier_size=max_frontier_size,
            reached_count=0,
            limit=max_depth,
            iterations=iteration_log,
        )

In [36]:
import pandas as pd

def show_results(results):
    rows = []

    for result in results:
        rows.append({
            "Algorithm": result.algorithm,
            "Status": result.status,
            "Solution depth": result.solution_depth,
            "Solution cost": result.solution_cost,
            "Nodes expanded": result.nodes_expanded,
            "Max frontier size": result.max_frontier_size,
            "Reached count": result.reached_count,
            "Limit": result.limit,
        })

    return pd.DataFrame(rows)

In [37]:
# TODO 10:
# Create your first custom map here.

custom_grid_1 = [
    [0, 0, 0, 0, 0],
    [1, 1, 1, 1, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1],
    [0, 0, 0, 0, 0],
]

custom_start_1 = (0, 0)
custom_goal_1 = (4, 4)

custom_problem_1 = GridProblem(custom_grid_1, custom_start_1, custom_goal_1)

bfs = BreadthFirstSearch()
dfs = DepthFirstSearch()
dls = DepthLimitedSearch()
ids = IterativeDeepeningSearch()

custom_results_1 = [
    bfs.search(custom_problem_1),
    dfs.search(custom_problem_1),
    dls.search(custom_problem_1, limit=20),
    ids.search(custom_problem_1, max_depth=40),
]

show_results(custom_results_1)

,Algorithm,Status,Solution depth,Solution cost,Nodes expanded,Max frontier size,Reached count,Limit
0,BFS,success,16,16,16,1,17,NaN
1,DFS,success,16,16,16,1,17,NaN
2,DLS,success,20,20,11374,21,0,20.0
3,IDS,success,16,16,26077,17,0,16.0


In [38]:
# TODO 11:
# Create your second custom map here.

custom_grid_2 = [
    # Replace this with your own grid.
]
# TODO 11:
# Create your second custom map here.

custom_grid_2 = [
    [0, 0, 1, 0, 0, 0],
    [1, 0, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 1],
    [1, 1, 1, 1, 0, 0],
]

custom_start_2 = (0, 0)
custom_goal_2 = (5, 5)

custom_problem_2 = GridProblem(custom_grid_2, custom_start_2, custom_goal_2)

custom_results_2 = [
    bfs.search(custom_problem_2),
    dfs.search(custom_problem_2),
    dls.search(custom_problem_2, limit=20),
    ids.search(custom_problem_2, max_depth=40),
]

show_results(custom_results_2)

,Algorithm,Status,Solution depth,Solution cost,Nodes expanded,Max frontier size,Reached count,Limit
0,BFS,success,12,12,20,3,23,NaN
1,DFS,success,16,16,21,3,23,NaN
2,DLS,success,20,20,1012,21,0,20.0
3,IDS,success,12,12,2988,13,0,12.0
